In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day03" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w7" / "day03"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
from sqlalchemy import select
from app.core.config import get_settings
from app.db.session import session_scope
from app.integrations.factory import get_embedder
from app.rag import store
from app.rag.retriever import expand_queries
settings = get_settings()
QUESTION = '1~2급 직원의 숙박비 상한은 얼마인가요?'

def candidates_for(question: str) -> list[dict]:
    embedder = get_embedder()
    with session_scope() as session:
        seen: dict[int, dict] = {}
        for one in expand_queries(question, settings.multi_query_n):
            for row in store.vector_search(session, embedder.embed_query(one), limit=settings.retrieval_top_k):
                seen.setdefault(row['chunk_id'], row)
        vector_rows = sorted(seen.values(), key=lambda r: r['score'], reverse=True)
        keyword_rows = store.keyword_search(session, question, limit=settings.retrieval_top_k)
    return store.fuse(vector_rows, keyword_rows)[:settings.retrieval_top_k]
rows = candidates_for(QUESTION)
for (n, row) in enumerate(rows, 1):
    if n == settings.rerank_top_n + 1:
        print(f"{'─' * 27} 여기서 잘립니다 (rerank_top_n = {settings.rerank_top_n})")
    print(f"{n:>2}  {row['doc_id']}  {row['version']}  {row['kind']:<2}  {row['locator'][:22]:<22} {row['score']:.3f}")

09:48:28 INFO     sentence_transformers.base.model: No device provided, using cpu
09:48:28 INFO     sentence_transformers.base.model: Loading SentenceTransformer model from C:\hw\hw\python_basic\models\bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

09:48:30 INFO     app.integrations.local_embed: 로컬 임베딩 모델 준비: C:\hw\hw\python_basic\models\bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 1  DOC-HR-014  v2.0  표   표2                     0.714
 2  DOC-HR-014  v1.1  표   표2                     0.690
 3  DOC-HR-014  v1.1  조항  제3장 제11조(숙박비) · 별표1    0.648
─────────────────────────── 여기서 잘립니다 (rerank_top_n = 3)
 4  DOC-HR-014  v2.0  조항  제3장 제2절 제14조(숙박비) · 별표 0.691
 5  DOC-HR-014  v2.0  표   표3                     0.647
 6  DOC-HR-014  v2.0  조항  제3장 제1절 제11조(일비) · 별표1 0.601
 7  DOC-HR-014  v1.1  조항  제3장 제8조(일비) · 별표1      0.594
 8  DOC-HR-014  v2.0  조항  제3장 제2절 제15조(지역 구분) ·  0.590
 9  DOC-HR-014  v2.0  조항  제3장 제2절 제16조(숙박비의 예외)  0.659
10  DOC-HR-014  v2.0  조항  제3장 제1절 제12조(식비) · 별표1 0.653
11  DOC-HR-014  v1.1  조항  제1장 제4조(여비의 종류) · 별표1  0.000
12  DOC-HR-014  v1.1  조항  제3장 제9조(식비)            0.643


In [3]:
from app.rag.reranker import rerank, score as lexical_score

before = [(r['doc_id'], r['locator'][:12], r['score'], lexical_score(QUESTION, r['text']), 0.03 if r['kind'] == '표' else 0.0) for r in rows]

after = rerank(QUESTION, rows, top_n=len(rows))

print('  [리랭킹 전]                                  [리랭킹 후]')
for (n, (left, right)) in enumerate(zip(before, after), 1):
    (doc_id, locator, vec, lex, bonus) = left
    print(f"{n:>2} {doc_id} {locator:<12} 벡터 {vec:.3f}   {n:>2} {right['doc_id']} {right['locator'][:12]:<12} rerank {right['rerank_score']:.3f}  (벡터 {right['score']:.3f})")


  [리랭킹 전]                                  [리랭킹 후]
 1 DOC-HR-014 표2           벡터 0.714    1 DOC-HR-014 표2           rerank 0.530  (벡터 0.714)
 2 DOC-HR-014 표2           벡터 0.690    2 DOC-HR-014 표2           rerank 0.513  (벡터 0.690)
 3 DOC-HR-014 제3장 제11조(숙박비 벡터 0.648    3 DOC-HR-014 제3장 제2절 제14조 rerank 0.484  (벡터 0.691)
 4 DOC-HR-014 제3장 제2절 제14조 벡터 0.691    4 DOC-HR-014 표3           rerank 0.483  (벡터 0.647)
 5 DOC-HR-014 표3           벡터 0.647    5 DOC-HR-014 제3장 제2절 제16조 rerank 0.461  (벡터 0.659)
 6 DOC-HR-014 제3장 제1절 제11조 벡터 0.601    6 DOC-HR-014 제3장 제1절 제12조 rerank 0.457  (벡터 0.653)
 7 DOC-HR-014 제3장 제8조(일비)  벡터 0.594    7 DOC-HR-014 제3장 제11조(숙박비 rerank 0.454  (벡터 0.648)
 8 DOC-HR-014 제3장 제2절 제15조 벡터 0.590    8 DOC-HR-014 제3장 제9조(식비)  rerank 0.450  (벡터 0.643)
 9 DOC-HR-014 제3장 제2절 제16조 벡터 0.659    9 DOC-HR-014 제3장 제1절 제11조 rerank 0.420  (벡터 0.601)
10 DOC-HR-014 제3장 제1절 제12조 벡터 0.653   10 DOC-HR-014 제3장 제8조(일비)  rerank 0.416  (벡터 0.594)
11 DOC-HR-014 제1장 제4조(여비의  벡터 0.000   11 DOC-HR-0